# Diagnostic UCB demonstration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kootru-repo/charge-filtered-cumulant-residuals/blob/main/notebooks/05_diagnostic_ucb_demo.ipynb)


Manuscript reference: Section IV, Proposition 1; full sample-complexity bookkeeping in `supplements/diagnostic_certificate_full.tex` of the manuscript working directory.

This notebook is a *protocol demonstration*: a single tiny example that shows the sample-split, Bonferroni-corrected upper-confidence-bound diagnostic on synthetic random-Pauli-shadow data, with one-sided coverage on a state where the truth is known. It is **not** a sample-complexity proof, **not** a full empirical-Bernstein implementation, and **not** evidence of measurement-protocol efficiency.

In [ ]:
# Bootstrap on Colab / fresh env (skipped if package + data are already present).
# Local users who cloned the repo and ran 'uv sync' skip the install entirely.
# On Colab the bootstrap clones the repo so data/ files are reachable via
# notebook-relative paths, chdir's into it, then installs the package with uv.
import importlib.util as _ilu
import pathlib as _pl
_HAS_PKG = _ilu.find_spec("connected_layer_sector") is not None
_HAS_DATA = _pl.Path("data").is_dir() or _pl.Path("../data").is_dir()
if not (_HAS_PKG and _HAS_DATA):
    import subprocess as _sp, os as _os
    _REPO = "/content/charge-filtered-cumulant-residuals"
    if not _pl.Path(_REPO).is_dir():
        _sp.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/kootru-repo/charge-filtered-cumulant-residuals.git",
            _REPO,
        ])
    _os.chdir(_REPO)
    # Pin uv to a known directory. UV_INSTALL_DIR is the installer's explicit
    # override; without it the location depends on CARGO_HOME / XDG_BIN_HOME
    # inheritance (Colab differs from local Linux).
    _UV_DIR = "/tmp/uv-bootstrap"
    _os.makedirs(_UV_DIR, exist_ok=True)
    _env = {**_os.environ, "UV_INSTALL_DIR": _UV_DIR}
    _sp.check_call(
        "curl -LsSf https://astral.sh/uv/install.sh | sh",
        shell=True, executable="/bin/bash", env=_env,
    )
    _UV = f"{_UV_DIR}/uv"
    _sp.check_call([_UV, "pip", "install", "--system", "-q", "-e", "."])

import connected_layer_sector
print(f"connected_layer_sector {connected_layer_sector.__version__} ready")


In [1]:
import math

import numpy as np

from connected_layer_sector import (
    deterministic_seed,
    determinant_state,
    word_moment,
)
from connected_layer_sector.operators import I2, X2, Y2, Z2, kron_chain

PAULIS = {"I": I2, "X": X2, "Y": Y2, "Z": Z2}

## Step 1: pick a known state and a target observable

Use a 4-orbital Slater determinant $|1, 1, 0, 0\rangle$. By Section V the catalog cumulants vanish on this state, so the truth for the cumulant we'll target ($\kappa_{[3]}(n_0, n_1, n_2; \rho)$) is exactly zero. The UCB protocol must produce a one-sided upper bound that covers $0$.

In [2]:
n_qubits = 4
rho = determinant_state(n_qubits, [1, 1, 0, 0])

# Target observables: the three single-site moments mu_i = <n_i> and the
# pair moments mu_ij and the triple mu_012. The cumulant
#   kappa_{[3]}(n_0, n_1, n_2; rho)
#     = mu_012 - mu_0 mu_12 - mu_1 mu_02 - mu_2 mu_01 + 2 mu_0 mu_1 mu_2
# is zero on this state (Sec V).
true_mu = {
    "0": float(np.real(word_moment(rho, (("n", 0),), n_qubits))),
    "1": float(np.real(word_moment(rho, (("n", 1),), n_qubits))),
    "2": float(np.real(word_moment(rho, (("n", 2),), n_qubits))),
    "01": float(np.real(word_moment(rho, (("n", 0), ("n", 1)), n_qubits))),
    "02": float(np.real(word_moment(rho, (("n", 0), ("n", 2)), n_qubits))),
    "12": float(np.real(word_moment(rho, (("n", 1), ("n", 2)), n_qubits))),
    "012": float(np.real(word_moment(rho, (("n", 0), ("n", 1), ("n", 2)), n_qubits))),
}
for k, v in true_mu.items():
    print(f"  mu_{k} = {v:.4f}")

true_kappa = (
    true_mu["012"]
    - true_mu["0"] * true_mu["12"]
    - true_mu["1"] * true_mu["02"]
    - true_mu["2"] * true_mu["01"]
    + 2 * true_mu["0"] * true_mu["1"] * true_mu["2"]
)
print(f"\ntrue kappa_{{[3]}}(n_0, n_1, n_2; rho) = {true_kappa:.2e}  (manuscript Sec V: 0)")

  mu_0 = 1.0000
  mu_1 = 1.0000
  mu_2 = 0.0000
  mu_01 = 1.0000
  mu_02 = 0.0000
  mu_12 = 0.0000
  mu_012 = 0.0000

true kappa_{[3]}(n_0, n_1, n_2; rho) = 0.00e+00  (manuscript Sec V: 0)


## Step 2: random-Pauli classical-shadow protocol

On each shot, pick a per-qubit Pauli basis uniformly at random, measure all qubits in that basis, and apply the inverse of the random-Pauli measurement channel:

$$\hat{\rho}_{\text{shot}} = \bigotimes_{q} \big(3 |b_q\rangle\langle b_q| - I\big)$$

where $|b_q\rangle$ is the measured $\pm 1$ eigenstate of the chosen Pauli at qubit $q$. Estimators for any subword moment are recovered by averaging $\Tr(\hat{\rho}_{\text{shot}} A_B)$ over shots.

In [3]:
rng = np.random.default_rng(seed=deterministic_seed("ucb_demo_n4_dna"))

PAULI_LABELS = ["X", "Y", "Z"]

def measure_one_shot(rho, n_qubits, rng):
    """Pick per-qubit Pauli basis uniformly; measure all qubits; return the
    chosen bases and the +/-1 outcomes."""
    bases = [PAULI_LABELS[k] for k in rng.integers(0, 3, size=n_qubits)]
    # Build the measurement basis by diagonalizing each chosen Pauli on its qubit
    # and sampling outcomes from the projection of rho.
    # Project rho through each qubit's measurement basis sequentially.
    # For simplicity here we use the eigenvectors of each Pauli: each Pauli has
    # eigenvalues +/-1 with corresponding eigenvectors.
    EIG = {
        "X": (np.array([1, 1]) / math.sqrt(2), np.array([1, -1]) / math.sqrt(2)),
        "Y": (np.array([1, 1j]) / math.sqrt(2), np.array([1, -1j]) / math.sqrt(2)),
        "Z": (np.array([1, 0]), np.array([0, 1])),
    }
    # Probability table for the 2^n basis outcomes in the chosen product basis.
    # Build the projector to each outcome and compute Tr(P rho).
    probs = np.zeros(2 ** n_qubits)
    for idx in range(2 ** n_qubits):
        bits = [(idx >> (n_qubits - 1 - q)) & 1 for q in range(n_qubits)]
        vecs = [EIG[bases[q]][bits[q]] for q in range(n_qubits)]
        psi = vecs[0]
        for v in vecs[1:]:
            psi = np.kron(psi, v)
        probs[idx] = float(np.real(psi.conj() @ rho @ psi))
    probs = np.clip(probs, 0.0, None)
    probs /= probs.sum()
    out_idx = int(rng.choice(2 ** n_qubits, p=probs))
    bits = [(out_idx >> (n_qubits - 1 - q)) & 1 for q in range(n_qubits)]
    outcomes = [+1 if b == 0 else -1 for b in bits]
    return bases, outcomes


def shadow_estimator_for_n_word(bases_list, outcomes_list, sites):
    """Estimator for the moment <n_{i_1} n_{i_2} ... n_{i_k}> = <prod_i (I-Z_i)/2>
    from a random-Pauli shadow record.

    For each shot, the shadow is a product of one-qubit shadows. The
    one-qubit estimator for a single-qubit operator O at qubit q is
    Tr(O (3 |b><b| - I)) = 3 <b|O|b> - tr(O), and for n_q = (I - Z)/2:

      contribution =
        if basis_q == 'Z':  3 * (1 if outcome=-1 else 0) - 1 = 2 if -1 else -1
        if basis_q != 'Z':  -1   (because <b|n_q|b> = 1/2 for X- or Y- eigenstates)
    """
    samples = []
    for bases, outcomes in zip(bases_list, outcomes_list):
        prod = 1.0
        for q in sites:
            if bases[q] == "Z":
                # n_q expectation in Z-basis: 1 if outcome=-1, else 0
                # one-qubit shadow estimator: 3 * <b|n|b> - tr(n) = 3*v - 1
                v = 1.0 if outcomes[q] == -1 else 0.0
                contrib = 3.0 * v - 1.0
            else:
                # X or Y basis: <b|n|b> = 1/2 for both eigenstates
                contrib = 3.0 * 0.5 - 1.0
            prod *= contrib
        samples.append(prod)
    samples = np.array(samples)
    return float(samples.mean()), float(samples.std(ddof=1))

## Step 3: collect a sample-split shadow record

Take $M = 4000$ total shots; split into $M_1 = 2000$ for the diagnostic and $M_2 = 2000$ reserved holdout.

In [4]:
M = 4000
bases_all, outcomes_all = [], []
for _ in range(M):
    bases, outcomes = measure_one_shot(rho, n_qubits, rng)
    bases_all.append(bases)
    outcomes_all.append(outcomes)

M1 = M // 2
bases_diag, outcomes_diag = bases_all[:M1], outcomes_all[:M1]
bases_hold, outcomes_hold = bases_all[M1:], outcomes_all[M1:]
print(f"shots: M = {M}, M_1 (diagnostic) = {M1}, M_2 (holdout) = {M - M1}")

shots: M = 4000, M_1 (diagnostic) = 2000, M_2 (holdout) = 2000


## Step 4: estimate the seven moments and the cumulant

Compute $\hat{\mu}_B$ on the diagnostic half for each subword $B$, then assemble the cumulant by the manuscript's length-3 partition formula:

$$\kappa_{123} = \mu_{123} - \mu_1 \mu_{23} - \mu_2 \mu_{13} - \mu_3 \mu_{12} + 2 \mu_1 \mu_2 \mu_3.$$

In [5]:
subwords = {
    "0":   (0,),
    "1":   (1,),
    "2":   (2,),
    "01":  (0, 1),
    "02":  (0, 2),
    "12":  (1, 2),
    "012": (0, 1, 2),
}

est_mu = {}
est_se = {}  # standard errors of the mean
for label, sites in subwords.items():
    mean, sd = shadow_estimator_for_n_word(bases_diag, outcomes_diag, sites)
    est_mu[label] = mean
    est_se[label] = sd / math.sqrt(M1)
    print(f"  hat_mu_{label} = {mean:+.4f} +/- {est_se[label]:.4f} (true: {true_mu[label]:+.4f})")

kappa_hat = (
    est_mu["012"]
    - est_mu["0"] * est_mu["12"]
    - est_mu["1"] * est_mu["02"]
    - est_mu["2"] * est_mu["01"]
    + 2 * est_mu["0"] * est_mu["1"] * est_mu["2"]
)
print(f"\nhat_kappa_{{[3]}} = {kappa_hat:+.4f}  (truth: {true_kappa:.2e})")

  hat_mu_0 = +0.9890 +/- 0.0157 (true: +1.0000)
  hat_mu_1 = +0.9882 +/- 0.0157 (true: +1.0000)
  hat_mu_2 = +0.0208 +/- 0.0156 (true: +0.0000)
  hat_mu_01 = +0.9715 +/- 0.0243 (true: +1.0000)
  hat_mu_02 = +0.0119 +/- 0.0191 (true: +0.0000)
  hat_mu_12 = +0.0205 +/- 0.0190 (true: +0.0000)
  hat_mu_012 = +0.0093 +/- 0.0231 (true: +0.0000)

hat_kappa_{[3]} = -0.0023  (truth: 0.00e+00)


## Step 5: Hoeffding one-sided UCB on the cumulant

Use a Hoeffding tail bound on each subword moment estimator. Each one-shot estimator for an $n$-letter $n_i n_j \cdots$ moment lies in $[-1, 2]^k$ at worst (each one-qubit shadow contribution is in $\{-1, +2\}$ for $Z$-bases and $\{-1, +0.5\}$ for $X/Y$-bases), so each per-shot sample is bounded; absolute Hoeffding range $R$ for a $k$-site moment is $\le 3^k$. Bonferroni-correct across the 7 subwords for an overall $1 - \alpha$ guarantee.

In [6]:
alpha = 0.05
n_subwords = len(subwords)
alpha_per = alpha / n_subwords

def hoeffding_radius(M_shots, R, alpha_per):
    """Symmetric Hoeffding radius for a sample mean of i.i.d. samples
    in [-R, R]: |hat - true| <= R sqrt(2 ln(2/alpha) / M)"""
    return R * math.sqrt(2 * math.log(2 / alpha_per) / M_shots)

rad = {}
for label, sites in subwords.items():
    R = 3 ** len(sites)  # absolute Hoeffding range
    rad[label] = hoeffding_radius(M1, R, alpha_per)
    print(f"  rad(mu_{label}) = {rad[label]:.4f}  (Hoeffding R = {R})")

# Propagate radii through the cumulant polynomial via
#   |kappa_hat - kappa| <= sum_terms |coefficient| * (radius bound for that term)
# and a coarse term-by-term Lipschitz argument: for products like
# mu_a * mu_b, the radius is |mu_b| * rad(mu_a) + |mu_a| * rad(mu_b).
def prod_radius(*pairs):
    """Propagate radii through a product of two or three terms via Lipschitz."""
    means = [m for m, r in pairs]
    rads = [r for m, r in pairs]
    out = 0.0
    for i in range(len(pairs)):
        contrib = rads[i]
        for j in range(len(pairs)):
            if j == i:
                continue
            contrib *= abs(means[j])
        out += contrib
    return out

rad_kappa = (
    rad["012"]
    + prod_radius((est_mu["0"], rad["0"]), (est_mu["12"], rad["12"]))
    + prod_radius((est_mu["1"], rad["1"]), (est_mu["02"], rad["02"]))
    + prod_radius((est_mu["2"], rad["2"]), (est_mu["01"], rad["01"]))
    + 2 * prod_radius(
        (est_mu["0"], rad["0"]),
        (est_mu["1"], rad["1"]),
        (est_mu["2"], rad["2"]),
    )
)
ucb = abs(kappa_hat) + rad_kappa
print(f"\n|hat_kappa| = {abs(kappa_hat):.4f}")
print(f"radius      = {rad_kappa:.4f}")
print(f"UCB         = {ucb:.4f}  (alpha = {alpha})")

  rad(mu_0) = 0.2252  (Hoeffding R = 3)
  rad(mu_1) = 0.2252  (Hoeffding R = 3)
  rad(mu_2) = 0.2252  (Hoeffding R = 3)
  rad(mu_01) = 0.6756  (Hoeffding R = 9)
  rad(mu_02) = 0.6756  (Hoeffding R = 9)
  rad(mu_12) = 0.6756  (Hoeffding R = 9)
  rad(mu_012) = 2.0268  (Hoeffding R = 27)

|hat_kappa| = 0.0023
radius      = 4.0613
UCB         = 4.0636  (alpha = 0.05)


## Step 6: coverage check

The truth is $\kappa_{[3]} = 0$ exactly. The one-sided UCB on $|\kappa|$ should cover $0$ (i.e., UCB $\ge 0$, which is trivially true) and the underlying assertion is that with probability $\ge 1 - \alpha$, $|\kappa_{\text{true}}| \le \text{UCB}$. Verify this on this single sample by checking $\text{UCB} \ge |\kappa_{\text{true}}| = 0$.

In [7]:
assert ucb >= abs(true_kappa), (
    f"UCB failed to cover truth: ucb={ucb:.4f}, |kappa_true|={abs(true_kappa):.2e}"
)
assert kappa_hat == kappa_hat  # not NaN
print("PASS: one-sided UCB covers the true kappa (Sec V zero baseline).")
print("PASS: kappa_hat is finite.")

PASS: one-sided UCB covers the true kappa (Sec V zero baseline).
PASS: kappa_hat is finite.


## Caveats and where the manuscript's full UCB lives

This is a single-sample, single-state, single-cumulant demonstration. It illustrates the three protocol steps (sample-split, per-subword estimator, Bonferroni-corrected radius propagation) but does NOT implement:

- The catalog-restricted UCB over many catalog words simultaneously (manuscript Lemma 2 catalog closure).
- Empirical-Bernstein refinements of the per-subword radius.
- The Jordan-Wigner shadow-range overhead $R_\star$ analysis for hopping and excitation operators.
- The full sample-complexity proof.

Those are in `supplements/diagnostic_certificate_full.tex` of the manuscript working directory and are out of scope for this reproducibility demo.